<img src="https://s8.hostingkartinok.com/uploads/images/2018/08/308b49fcfbc619d629fe4604bceb67ac.jpg" width=500, height=450>
<h3 style="text-align: center;"><b>Физтех-Школа Прикладной математики и информатики (ФПМИ) МФТИ</b></h3>

---

***Some parts of the notebook are almost the copy of [ mmta-team course](https://github.com/mmta-team/mmta_fall_2020). Special thanks to mmta-team for making them publicly available. [Original notebook](https://github.com/mmta-team/mmta_fall_2020/blob/master/tasks/01_word_embeddings/task_word_embeddings.ipynb).***

<b> Прочитайте семинар, пожалуйста, для успешного выполнения домашнего задания. В конце ноутка напишите свой вывод. Работа без вывода оценивается ниже.

## Задача поиска схожих по смыслу предложений

Мы будем ранжировать вопросы [StackOverflow](https://stackoverflow.com) на основе семантического векторного представления

До этого в курсе не было речи про задачу ранжировния, поэтому введем математическую формулировку

## Задача ранжирования(Learning to Rank)

* $X$ - множество объектов
* $X^l = \{x_1, x_2, ..., x_l\}$ - обучающая выборка
<br>На обучающей выборке задан порядок между некоторыми элементами, то есть нам известно, что некий объект выборки более релевантный для нас, чем другой:
* $i \prec j$ - порядок пары индексов объектов на выборке $X^l$ c индексами $i$ и $j$
### Задача:
построить ранжирующую функцию $a$ : $X \rightarrow R$ такую, что
$$i \prec j \Rightarrow a(x_i) < a(x_j)$$

<img src="https://d25skit2l41vkl.cloudfront.net/wp-content/uploads/2016/12/Featured-Image.jpg" width=500, height=450>

### Embeddings

Будем использовать предобученные векторные представления слов на постах Stack Overflow.<br>
[A word2vec model trained on Stack Overflow posts](https://github.com/vefstathiou/SO_word2vec)

In [ ]:
!wget https://zenodo.org/record/1199620/files/SO_vectors_200.bin?download=1

In [ ]:
!pip install gensim

In [ ]:
from gensim.models.keyedvectors import KeyedVectors
wv_embeddings = KeyedVectors.load_word2vec_format("SO_vectors_200.bin?download=1", binary=True)

#### Как пользоваться этими векторами?

Посмотрим на примере одного слова, что из себя представляет embedding

In [ ]:
word = 'dog'
if word in wv_embeddings:
    print(wv_embeddings[word].dtype, wv_embeddings[word].shape)

In [ ]:
print(f"Num of words: {len(wv_embeddings.index_to_key)}")

Найдем наиболее близкие слова к слову `dog`:

#### ***Вопрос 1:***
* Входит ли слово `cat` в топ-5 близких слов к слову `dog`? Какое место оно занимает?


In [ ]:
wv_embeddings.most_similar("dog", topn=5)

In [ ]:
# method most_simmilar
print("Yes" if "cat" in [word for word, _ in wv_embeddings.most_similar("dog", topn=5)] else "No")

***Ваш ответ:*** 'cat' не входит в топ-5 ближайших слов к 'dog', однако слово "cats" входит в этот список.

### Векторные представления текста

Перейдем от векторных представлений отдельных слов к векторным представлениям вопросов, как к **среднему** векторов всех слов в вопросе. Если для какого-то слова нет предобученного вектора, то его нужно пропустить. Если вопрос не содержит ни одного известного слова, то нужно вернуть нулевой вектор.

In [ ]:
import numpy as np
import re
# you can use your tokenizer
# for example, from nltk.tokenize import WordPunctTokenizer
class MyTokenizer:
    def __init__(self):
        pass
    def tokenize(self, text):
        return re.findall(r'\w+', text)
tokenizer = MyTokenizer()

In [ ]:
def question_to_vec(question, embeddings, tokenizer, dim=200):
    """
        question: строка
        embeddings: наше векторное представление
        dim: размер любого вектора в нашем представлении

        return: векторное представление для вопроса
    """
    tokens = tokenizer.tokenize(question.lower())
    embeddings_q = [embeddings[word] for word in tokens if word in embeddings]
    if embeddings_q:
      return np.mean(embeddings_q, axis = 0)
    else:
      return np.zeros(dim, dtype = np.float32)



Теперь у нас есть метод для создания векторного представления любого предложения.

#### ***Вопрос 2:***

* Какая третья (с индексом 2) компонента вектора предложения `I love neural networks` (округлите до 2 знаков после запятой)?

In [ ]:
# Предложение
question = "I love neural networks"
question_em = question_to_vec(question, wv_embeddings, tokenizer)
print(f"Третья компонента: {round(float(question_em[2]), 2)}")

**Ответ:** Третья Компонента равна -1.29 (после округления)

### Оценка близости текстов

Представим, что мы используем идеальные векторные представления слов. Тогда косинусное расстояние между дублирующими предложениями должно быть меньше, чем между случайно взятыми предложениями.

Сгенерируем для каждого из $N$ вопросов $R$ случайных отрицательных примеров и примешаем к ним также настоящие дубликаты. Для каждого вопроса будем ранжировать с помощью нашей модели $R + 1$ примеров и смотреть на позицию дубликата. Мы хотим, чтобы дубликат был первым в ранжированном списке.

#### Hits@K
Первой простой метрикой будет количество корректных попаданий для какого-то $K$:
$$ \text{Hits@K} = \frac{1}{N}\sum_{i=1}^N \, [rank\_q_i^{'} \le K],$$
* $\begin{equation*}
[x < 0 ] \equiv
 \begin{cases}
   1, &x < 0\\
   0, &x \geq 0
 \end{cases}
\end{equation*}$ - индикаторная функция
* $q_i$ - $i$-ый вопрос
* $q_i^{'}$ - его дубликат
* $rank\_q_i^{'}$ - позиция дубликата в ранжированном списке ближайших предложений для вопроса $q_i$.

Hits@K  измеряет долю вопросов, для которых правильный ответ попал в топ-K позиций среди отранжированных кандидатов.

#### DCG@K
Второй метрикой будет упрощенная DCG метрика, учитывающая порядок элементов в списке путем домножения релевантности элемента на вес равный обратному логарифму номера позиции::
$$ \text{DCG@K} = \frac{1}{N} \sum_{i=1}^N\frac{1}{\log_2(1+rank\_q_i^{'})}\cdot[rank\_q_i^{'} \le K],$$
С такой метрикой модель штрафуется за большой ранк корректного ответа.

DCG@K  измеряет качество ранжирования, учитывая не только факт наличия правильного ответа в топ-K, но и ***его точную позицию***.

<img src='https://hsto.org/files/1c5/edf/dee/1c5edfdeebce4b71a86bdf986d9f88f2.jpg' width=400, height=200>

#### Пример оценок

Вычислим описанные выше метрики для игрушечного примера.
Пусть
* $N = 1$, $R = 3$
* <font color='green'>"Что такое python?"</font> - вопрос $q_1$
* <font color='red'>"Что такое язык python?"</font> - его дубликат $q_i^{'}$

Пусть модель выдала следующий ранжированный список кандидатов:

1. "Как изучить с++?"
2. <font color='red'>"Что такое язык python?"</font>
3. "Хочу учить Java"
4. "Не понимаю Tensorflow"

$\Rightarrow rank\_q_i^{'} = 2$

Вычислим метрику *Hits@K* для *K = 1, 4*:

- [K = 1] $\text{Hits@1} =  [rank\_q_i^{'} \le 1]$

Проверяем условие $ \text{rank}_{q'_1} \leq 1 $: ***условие неверно***.

Следовательно, $[\text{rank}_{q'_1} \leq 1] = 0$.

- [K = 4] $\text{Hits@4} =  [rank\_q_i^{'} \le 4] = 1$

Проверяем условие $ \text{rank}_{q'_1} \leq 4 $: ***условие верно***.

Вычислим метрику *DCG@K* для *K = 1, 4*:
- [K = 1] $\text{DCG@1} = \frac{1}{\log_2(1+2)}\cdot[2 \le 1] = 0$
- [K = 4] $\text{DCG@4} = \frac{1}{\log_2(1+2)}\cdot[2 \le 4] = \frac{1}{\log_2{3}}$

#### ***Вопрос 3***:
* Вычислите `DCG@10`, если $rank\_q_i^{'} = 9$(округлите до одного знака после запятой):

  [K = 10] $\text{DCG@10} = \frac{1}{\log_2(1+9)}\cdot[9 \le 10] = \frac{1}{\log_2{10}} ≈ 0.3$



#### Более сложный пример оценок

Рассмотрим пример с $ N > 1 $, где $ N = 3 $ (три вопроса) и для каждого вопроса заданы позиции их дубликатов. Вычислим метрики **Hits@K** для разных значений $ K $.

---

- $ N = 3 $: Три вопроса ($ q_1, q_2, q_3 $).
- Для каждого вопроса известна позиция его дубликата ($ \text{rank}_{q'_i} $):
  - $ \text{rank}_{q'_1} = 2 $,
  - $ \text{rank}_{q'_2} = 5 $,
  - $ \text{rank}_{q'_3} = 1 $.

Мы будем вычислять **Hits@K** для $ K = 1, 5 $.

---

**Для $ K = 1 $:**

Подставим значения:
$$
\text{Hits@1} = \frac{1}{3} \cdot \left( [\text{rank}_{q'_1} \leq 1] + [\text{rank}_{q'_2} \leq 1] + [\text{rank}_{q'_3} \leq 1] \right).
$$

Проверяем условие $ \text{rank}_{q'_i} \leq 1 $ для каждого вопроса:
- $ \text{rank}_{q'_1} = 2 $ → $ 2 \not\leq 1 $ → $ 0 $,
- $ \text{rank}_{q'_2} = 5 $ → $ 5 \not\leq 1 $ → $ 0 $,
- $ \text{rank}_{q'_3} = 1 $ → $ 1 \leq 1 $ → $ 1 $.

Сумма:
$$
\text{Hits@1} = \frac{1}{3} \cdot (0 + 0 + 1) = \frac{1}{3}.
$$

$$
\boxed{\text{Hits@1} = \frac{1}{3}}.
$$

---

**Для $ K = 5 $:**

Подставим значения:
$$
\text{Hits@5} = \frac{1}{3} \cdot \left( [\text{rank}_{q'_1} \leq 5] + [\text{rank}_{q'_2} \leq 5] + [\text{rank}_{q'_3} \leq 5] \right).
$$

Проверяем условие $ \text{rank}_{q'_i} \leq 5 $ для каждого вопроса:
- $ \text{rank}_{q'_1} = 2 $ → $ 2 \leq 5 $ → $ 1 $,
- $ \text{rank}_{q'_2} = 5 $ → $ 5 \leq 5 $ → $ 1 $,
- $ \text{rank}_{q'_3} = 1 $ → $ 1 \leq 5 $ → $ 1 $.

Сумма:
$$
\text{Hits@5} = \frac{1}{3} \cdot (1 + 1 + 1) = 1.
$$

$$
\boxed{\text{Hits@5} = 1}.
$$

Теперь вычислим метрику **DCG@K** для того же примера, где $ N = 3 $ (три вопроса), и для каждого вопроса известна позиция его дубликата ($ \text{rank}_{q'_i} $):

- $ \text{rank}_{q'_1} = 2 $,
- $ \text{rank}_{q'_2} = 5 $,
- $ \text{rank}_{q'_3} = 1 $.

Мы будем вычислять **DCG@K** для $ K = 1, 5 $.

---
**Для $ K = 1 $:**
Подставим значения:
$$
\text{DCG@1} = \frac{1}{3} \cdot \left( \frac{1}{\log_2(1 + \text{rank}_{q'_1})} \cdot [\text{rank}_{q'_1} \leq 1] + \frac{1}{\log_2(1 + \text{rank}_{q'_2})} \cdot [\text{rank}_{q'_2} \leq 1] + \frac{1}{\log_2(1 + \text{rank}_{q'_3})} \cdot [\text{rank}_{q'_3} \leq 1] \right).
$$

Проверяем условие $ \text{rank}_{q'_i} \leq 1 $ для каждого вопроса:
- $ \text{rank}_{q'_1} = 2 $ → $ 2 \not\leq 1 $ → $ 0 $,
- $ \text{rank}_{q'_2} = 5 $ → $ 5 \not\leq 1 $ → $ 0 $,
- $ \text{rank}_{q'_3} = 1 $ → $ 1 \leq 1 $ → $ 1 $.

Сумма:
$$
\text{DCG@1} = \frac{1}{3} \cdot (0 + 0 + 1) = \frac{1}{3}.
$$
$$
\boxed{\text{DCG@1} = \frac{1}{3}}.
$$

---


**Для $ K = 5 $:**
Подставим значения:
$$
\text{DCG@5} = \frac{1}{3} \cdot \left( \frac{1}{\log_2(1 + \text{rank}_{q'_1})} \cdot [\text{rank}_{q'_1} \leq 5] + \frac{1}{\log_2(1 + \text{rank}_{q'_2})} \cdot [\text{rank}_{q'_2} \leq 5] + \frac{1}{\log_2(1 + \text{rank}_{q'_3})} \cdot [\text{rank}_{q'_3} \leq 5] \right).
$$

Проверяем условие $ \text{rank}_{q'_i} \leq 5 $ для каждого вопроса:
- $ \text{rank}_{q'_1} = 2 $ → $ 2 \leq 5 $ → $ 1 $,
- $ \text{rank}_{q'_2} = 5 $ → $ 5 \leq 5 $ → $ 1 $,
- $ \text{rank}_{q'_3} = 1 $ → $ 1 \leq 5 $ → $ 1 $.

Сумма:
$$
\text{DCG@5} = \frac{1}{3} \cdot (0.631 + 0.387 + 1) = \frac{1}{3} \cdot 2.018 \approx 0.673.
$$

$$
\boxed{\text{DCG@5} \approx 0.673}.
$$

#### ***Вопрос 4:***
* Найдите максимум `Hits@47 - DCG@1`?
Для предыдущего примера с рангами (2, 5, 1):

1. $\text{Hits@47}$ = 1
2. $\text{DCG@1}$ = $\frac{1}{3}$
3. Ответ: $\text{Hits@47} - \text{DCG@1} ≈ 0.67$



### HITS\_COUNT и DCG\_SCORE

Каждая функция имеет два аргумента: $dup\_ranks$ и $k$.

$dup\_ranks$ является списком, который содержит рейтинги дубликатов (их позиции в ранжированном списке).

К примеру для <font color='red'>"Что такое язык python?"</font> $dup\_ranks = [2]$.

In [ ]:
def hits_count(dup_ranks, k):
    """
        dup_ranks: list индексов дубликатов
        k: пороговое значение для ранга
        result: вернуть Hits@k
    """
    # Подсчитываем количество дубликатов, чей ранг <= k
    hits_value = sum(1 for dup in dup_ranks if dup <= k) / len(dup_ranks)
    return hits_value

In [ ]:
dup_ranks = [2]

k = 1
hits_value = hits_count(dup_ranks, k)
print(f"Hits@1 = {hits_value}")

k = 4
hits_value = hits_count(dup_ranks, k)
print(f"Hits@4 = {hits_value}")

In [ ]:
import math

def dcg_score(dup_ranks, k):
    """
        dup_ranks: list индексов дубликатов
        k: пороговое значение для ранга
        result: вернуть DCG@k
    """
    # Вычисляем сумму для всех релевантных дубликатов
    first_part = sum(( 1 / math.log2(1 + dup) for dup in dup_ranks if dup <= k))

    # Делим на общее количество вопросов
    dcg_value = first_part / len(dup_ranks)
    return dcg_value

In [ ]:
# Пример списка позиций дубликатов
dup_ranks = [2]

# Вычисляем DCG@1
dcg_value = dcg_score(dup_ranks, k=1)
print(f"DCG@1 = {dcg_value:.3f}")

# Вычисляем DCG@4
dcg_value = dcg_score(dup_ranks, k=4)
print(f"DCG@4 = {dcg_value:.3f}")

Протестируем функции. Пусть $N = 1$, то есть один эксперимент. Будем искать копию вопроса и оценивать метрики.

In [ ]:
import pandas as pd

In [ ]:
copy_answers = ["How does the catch keyword determine the type of exception that was thrown",]

# наши кандидаты
candidates_ranking = [["How Can I Make These Links Rotate in PHP",
                       "How does the catch keyword determine the type of exception that was thrown",
                       "NSLog array description not memory address",
                       "PECL_HTTP not recognised php ubuntu"],]

# dup_ranks — позиции наших копий, так как эксперимент один, то этот массив длины 1
dup_ranks = [2]

# вычисляем метрику для разных k
print('Ваш ответ HIT:', [hits_count(dup_ranks, k) for k in range(1, 5)])
print('Ваш ответ DCG:', [round(dcg_score(dup_ranks, k), 5) for k in range(1, 5)])

У вас должно получиться

In [ ]:
# correct_answers - метрика для разных k
correct_answers = pd.DataFrame([[0, 1, 1, 1], [0, 1 / (np.log2(3)), 1 / (np.log2(3)), 1 / (np.log2(3))]],
                               index=['HITS', 'DCG'], columns=range(1,5))
print(correct_answers)

### Данные
[arxiv link](https://drive.google.com/file/d/1QqT4D0EoqJTy7v9VrNCYD-m964XZFR7_/edit)

`train.tsv` - выборка для обучения.<br> В каждой строке через табуляцию записаны: **<вопрос>, <похожий вопрос>**

`validation.tsv` - тестовая выборка.<br> В каждой строке через табуляцию записаны: **<вопрос>, <похожий вопрос>, <отрицательный пример 1>, <отрицательный пример 2>, ...**

In [ ]:
def read_corpus(filename):
    data = []
    with open(filename, encoding='utf-8') as file:
        for line in file:
            data.append(line.strip().split('\t'))
    return data

Нам понадобиться только файл validation.

In [ ]:
validation_data = read_corpus('/kaggle/input/datasets/kite121/nlp-help/validation.tsv')

Кол-во строк

In [ ]:
len(validation_data)

Размер нескольких первых строк

In [ ]:
for i in range(25):
    print(i + 1, len(validation_data[i]))

### Ранжирование без обучения

Реализуйте функцию ранжирования кандидатов на основе косинусного расстояния. Функция должна по списку кандидатов вернуть отсортированный список пар (позиция в исходном списке кандидатов, кандидат). При этом позиция кандидата в полученном списке является его рейтингом (первый - лучший). Например, если исходный список кандидатов был [a, b, c], и самый похожий на исходный вопрос среди них - c, затем a, и в конце b, то функция должна вернуть список **[(2, c), (0, a), (1, b)]**.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from copy import deepcopy

In [ ]:
def rank_candidates(question, candidates, embeddings, tokenizer, dim=200):
    """
        question: строка
        candidates: массив строк(кандидатов) [a, b, c]
        result: пары (начальная позиция, кандидат) [(2, c), (0, a), (1, b)]
    """
    vec_q = question_to_vec(question, embeddings, tokenizer, dim)

    result = []
    for i, cand in enumerate(candidates):
        vec_c = question_to_vec(cand, embeddings, tokenizer, dim)
        val = cosine_similarity(vec_q.reshape(1, -1), vec_c.reshape(1, -1))[0][0]
        result.append((i, cand, val))

    result.sort(key=lambda x: x[2], reverse=True)
    result = [(i, cand) for i, cand, _ in result]
    return result

Протестируйте работу функции на примерах ниже. Пусть $N=2$, то есть два эксперимента

In [ ]:
questions = ['converting string to list', 'Sending array via Ajax fails']

candidates = [['Convert Google results object (pure js) to Python object', # первый эксперимент
               'C# create cookie from string and send it',
               'How to use jQuery AJAX for an outside domain?'],

              ['Getting all list items of an unordered list in PHP',      # второй эксперимент
               'WPF- How to update the changes in list item of a list',
               'select2 not displaying search results']]

In [ ]:
for question, q_candidates in zip(questions, candidates):
        ranks = rank_candidates(question, q_candidates, wv_embeddings, tokenizer)
        print(ranks)
        print()

Для первого экперимента вы можете полностью сравнить ваши ответы и правильные ответы. Но для второго эксперимента два ответа на кандидаты будут <b>скрыты</b>(*)

In [ ]:
# должно вывести
results = [[(1, 'C# create cookie from string and send it'),
            (0, 'Convert Google results object (pure js) to Python object'),
            (2, 'How to use jQuery AJAX for an outside domain?')],
           [(0, 'Getting all list items of an unordered list in PHP'), #скрыт
            (2, 'select2 not displaying search results'), #скрыт
            (1, 'WPF- How to update the changes in list item of a list')]] #скрыт

Последовательность начальных индексов вы должны получить `для эксперимента 1`  1, 0, 2.

#### ***Вопрос 5:***
* Какую последовательность начальных индексов вы получили `для эксперимента 2`(перечисление без запятой и пробелов, например, `102` для первого эксперимента?

Ответ: `021`


Теперь мы можем оценить качество нашего метода. Запустите следующие два блока кода для получения результата. Обратите внимание, что вычисление расстояния между векторами занимает некоторое время (примерно 10 минут). Можете взять для validation 1000 примеров.

In [ ]:
from tqdm.notebook import tqdm

In [ ]:
wv_ranking = []
max_validation_examples = 1000
for i, line in enumerate(tqdm(validation_data)):
    if i == max_validation_examples:
        break
    q, *ex = line
    ranks = rank_candidates(q, ex, wv_embeddings, tokenizer)
    wv_ranking.append([r[0] for r in ranks].index(0) + 1)

In [ ]:
for k in tqdm([1, 5, 10, 100, 500, 1000]):
    print("DCG@%4d: %.3f | Hits@%4d: %.3f" % (k, dcg_score(wv_ranking, k), k, hits_count(wv_ranking, k)))

Из формул выше можно понять, что

- $ \text{Hits@K} $ **монотонно неубывающая функция** $ K $, которая стремится к 1 при $ K \to \infty $.

- $ \text{DCG@K} $ **монотонно неубывающая функция** $ K $, но рост замедляется с увеличением $ K $ из-за убывания веса $ \frac{1}{\log_2(1 + \text{rank}_{q'_i})} $.

### Эмбеддинги, обученные на корпусе похожих вопросов

In [ ]:
train_data = read_corpus('/kaggle/input/datasets/kite121/nlp-help/train.tsv')

Улучшите качество модели.<br>Склеим вопросы в пары и обучим на них модель Word2Vec из gensim. Выберите размер window. Объясните свой выбор.

***Рассмотрим подробнее*** данное склеивание.

1. Каждая строка из train_data разбивается на вопрос (question) и список кандидатов.

2. Для каждого кандидата вопрос склеивается с ним в одну строку.

3. Склеенная строка (combined_text) токенизируется, и полученный список токенов добавляется в общий корпус (corpus).

***Пример***

    Вопрос: "What is Python?"
    Кандидаты: ["Python is a programming language", "Java is another language"]
    Склеенные строки:
        "What is Python? Python is a programming language"
        "What is Python? Java is another language"
         
    Токенизированные списки:
        ['what', 'is', 'python', 'python', 'is', 'a', 'programming', 'language']
        ['what', 'is', 'python', 'java', 'is', 'another', 'language']
         
     

In [ ]:
train_data[123132]

In [ ]:
corpus = []
for line in train_data:
    q, *cand = line
    for can in cand:
        combined_text = (q + " " + can).lower()
        tokens = tokenizer.tokenize(combined_text)
        corpus.append(tokens)


In [ ]:
from gensim.models import Word2Vec
embeddings_trained = Word2Vec(
    sentences=corpus,        # Корпус токенизированных текстов
    vector_size=200,         # Размерность векторов
    window=5,                # Размер окна контекста
    min_count=1,             # Минимальная частота слов
    workers=4                # Количество потоков
).wv

- Было выбрано window = 5, чтобы улавливался контекст вопроса и ответа. Если поставить window слишком маленьким, то контекст ответа с вопросом не будет виден. Если выбрать window слишком большим, то будет  много слов, и связи будут теряться.

- Был выбран min_count = 1, поскольку в тексте присутсвуют редкие, но важные слова, а сам корпус небольшой.

In [ ]:
wv_ranking = []
max_validation_examples = 1000
for i, line in enumerate(tqdm(validation_data)):
    if i == max_validation_examples:
        break
    q, *ex = line
    ranks = rank_candidates(q, ex, embeddings_trained, tokenizer)
    wv_ranking.append([r[0] for r in ranks].index(0) + 1)

In [ ]:
for k in tqdm([1, 5, 10, 100, 500, 1000]):
    print("DCG@%4d: %.3f | Hits@%4d: %.3f" % (k, dcg_score(wv_ranking, k), k, hits_count(wv_ranking, k)))

### Замечание:
Решить эту задачу с помощью обучения полноценной нейронной сети будет вам предложено, как часть задания в одной из домашних работ по теме "Диалоговые системы".

In [ ]:
!pip install spacy

In [1]:
import nltk
nltk.download("stopwords")

!python -m spacy download en_core_web_sm

[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 71.3 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from nltk.tokenize import WordPunctTokenizer, TreebankWordTokenizer
import spacy
from nltk.corpus import stopwords
import time

In [ ]:
EN_STOPWORDS = set(stopwords.words("english"))
regex_tokenizer = MyTokenizer()
wordpunct_tokenizer = WordPunctTokenizer()
treebank_tokenizer = TreebankWordTokenizer()

In [ ]:
spacy_nlp = spacy.load("en_core_web_sm")

In [ ]:
def tokenize_text(text, mode="regex"):
    text = text.lower()
    if mode == "regex":
        return regex_tokenizer.tokenize(text)
    if mode == "wordpunct":
        return [t.lower() for t in wordpunct_tokenizer.tokenize(text)]
    if mode == "treebank":
        return [t.lower() for t in treebank_tokenizer.tokenize(text)]
    raise ValueError(f"Unknown mode: {mode}")

In [ ]:
def normalize_tokens(tokens, remove_stopwords=False, lemmatize=None):
    toks = tokens
    if lemmatize == "spacy":
        doc = spacy_nlp(" ".join(toks))
        toks = [t.lemma_.lower() for t in doc if t.lemma_.strip()]
    if remove_stopwords:
        toks = [t for t in toks if t not in EN_STOPWORDS]
    toks = [t for t in toks if t and any(ch.isalnum() for ch in t)]
    return toks

In [ ]:
def question_to_vec_with_preproc(
    question,
    embeddings,
    dim=200,
    token_mode="regex",
    remove_stopwords=False,
    lemmatize=None
):
    tokens = tokenize_text(question, mode=token_mode)
    tokens = normalize_tokens(tokens, remove_stopwords=remove_stopwords, lemmatize=lemmatize)
    vectors = [embeddings[t] for t in tokens if t in embeddings]
    if not vectors:
        return np.zeros(dim, dtype=np.float32)
    return np.mean(vectors, axis=0)

In [ ]:
def rank_candidates_with_preproc(
    question,
    candidates,
    embeddings,
    dim=200,
    token_mode="regex",
    remove_stopwords=False,
    lemmatize=None
):
    q_vec = question_to_vec_with_preproc(
        question, embeddings, dim=dim,
        token_mode=token_mode, remove_stopwords=remove_stopwords, lemmatize=lemmatize
    )
    scored = []
    for i, cand in enumerate(candidates):
        c_vec = question_to_vec_with_preproc(
            cand, embeddings, dim=dim,
            token_mode=token_mode, remove_stopwords=remove_stopwords, lemmatize=lemmatize
        )
        score = cosine_similarity(q_vec.reshape(1, -1), c_vec.reshape(1, -1))[0][0]
        scored.append((i, cand, score))
    scored.sort(key=lambda x: x[2], reverse=True)
    return [(i, c) for i, c, _ in scored]

In [ ]:
def evaluate_pipeline(
    validation_data,
    embeddings,
    hits_count_fn,
    dcg_score_fn,
    token_mode="regex",
    remove_stopwords=False,
    lemmatize=None,
    max_examples=1000
):
    ranks_all = []
    started = time.time()

    for i, line in enumerate(validation_data):
        if i >= max_examples:
            break
        q, *cands = line
        ranked = rank_candidates_with_preproc(
            q, cands, embeddings,
            token_mode=token_mode,
            remove_stopwords=remove_stopwords,
            lemmatize=lemmatize
        )
        ranks_all.append([r[0] for r in ranked].index(0) + 1)

    elapsed = time.time() - started
    return {
        "DCG@1": dcg_score_fn(ranks_all, 1),
        "DCG@5": dcg_score_fn(ranks_all, 5),
        "DCG@10": dcg_score_fn(ranks_all, 10),
        "DCG@100": dcg_score_fn(ranks_all, 100),
        "Hits@1": hits_count_fn(ranks_all, 1),
        "Hits@5": hits_count_fn(ranks_all, 5),
        "Hits@10": hits_count_fn(ranks_all, 10),
        "Hits@100": hits_count_fn(ranks_all, 100),
        "time_sec": elapsed,
        "evaluated_examples": len(ranks_all),
    }

In [ ]:
max_validation_examples = 1000
configs = [
    {"name": "A_regex_baseline", "token_mode": "regex", "remove_stopwords": False, "lemmatize": None},
    {"name": "B_wordpunct", "token_mode": "wordpunct", "remove_stopwords": False, "lemmatize": None},
    {"name": "C_treebank", "token_mode": "treebank", "remove_stopwords": False, "lemmatize": None},
    {"name": "D_regex_plus_stopwords", "token_mode": "regex", "remove_stopwords": True, "lemmatize": None},
    {"name": "E_regex_plus_spacy_lemma", "token_mode": "regex", "remove_stopwords": False, "lemmatize": "spacy"},
    {"name": "F_regex_plus_stopwords_plus_spacy_lemma", "token_mode": "regex", "remove_stopwords": True, "lemmatize": "spacy"},
]

In [ ]:
from copy import deepcopy
from tqdm.auto import tqdm
import pandas as pd

rows = []
for cfg in tqdm(configs, desc="Evaluating configs"):
    row = deepcopy(cfg)
    m = evaluate_pipeline(
        validation_data=validation_data,
        embeddings=wv_embeddings,  # swap to embeddings_trained if needed
        hits_count_fn=hits_count,
        dcg_score_fn=dcg_score,
        token_mode=cfg["token_mode"],
        remove_stopwords=cfg["remove_stopwords"],
        lemmatize=cfg["lemmatize"],
        max_examples=max_validation_examples,
    )
    row.update(m)
    row["status"] = "OK"
    rows.append(row)

In [ ]:
results_df = pd.DataFrame(rows)
sort_cols = [c for c in ["DCG@10", "Hits@10"] if c in results_df.columns]
if sort_cols:
    results_df = results_df.sort_values(sort_cols, ascending=False)

In [ ]:
print(results_df)

In [ ]:
from pathlib import Path

out_dir = Path("/kaggle/working")
out_dir.mkdir(parents=True, exist_ok=True)

out_file = out_dir / "my_dataframe.csv"
results_df.to_csv(out_file, index=False)

print(f"Saved to: {out_file}")

Напишите свой вывод о полученных результатах.
* Какой принцип токенизации даёт качество лучше и почему?
* Помогает ли нормализация слов?
* Какие эмбеддинги лучше справляются с задачей и почему?
* Почему получилось плохое качество решения задачи?
* Предложите свой подход к решению задачи.

## Вывод: 
По результатам эксперимента лучшее качество показала базовая токенизация regex с удалением stopwords,
но улучшение относительно baseline очень небольшое. WordPunctTokenizer дал практически те же результаты,
а TreebankTokenizer оказался немного хуже. Нормализация в виде удаления stopwords слегка помогает,
а лемматизация через spaCy не улучшила качество и при этом сильно увеличила время работы.

Если сравнивать эмбеддинги, то предобученные StackOverflow embeddings оказались заметно лучше,
чем Word2Vec, обученный на нашем корпусе пар вопросов. Например, для предобученных эмбеддингов
DCG@10 = 0.525 и Hits@10 = 0.651, тогда как для обученных с нуля получилось DCG@10 = 0.412
и Hits@10 = 0.527. Это можно объяснить тем, что готовые эмбеддинги были обучены на гораздо
большем корпусе и лучше отражают семантику технических текстов.

В целом качество решения остаётся ограниченным, потому что вопрос представляется как среднее
векторов слов. Такой подход не учитывает порядок слов, различие в важности слов и плохо
передаёт смысл длинных технических вопросов. Для улучшения качества можно попробовать более
аккуратную предобработку текста, отдельно подобрать параметры Word2Vec (например, window,
vector_size, min_count, количество эпох обучения), а также обучать эмбеддинги на большем
объёме данных. Ещё один разумный вариант — не усреднять все слова одинаково, а сильнее
учитывать важные технические термины и слабее учитывать слишком частотные слова.

